# Постановка задачи и подготовка признаков

После первичного аудита проверяем схему, фиксируем роли полей и считаем
подозрительные погодные значения на всем CSV. Словарь всех полей:
[`data_dictionary.md`](../docs/data_dictionary.md).

Задача – ретроспективная классификация `has_ga`. Единица наблюдения – запись
о посадке, возможно с несколькими попытками захода. Ранний прогноз не заявляем.
В этом ноутбуке пока не обучаем модели и не создаем окончательный split.


In [1]:
# Используем пути относительно проекта, без привязки к компьютеру
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

data_path = project_root / 'data' / 'raw' / 'go_arounds_augmented.csv.gz'
dictionary_path = project_root / 'docs' / 'data_dictionary.md'
if not data_path.is_file() or not dictionary_path.is_file():
    raise FileNotFoundError('Нужны исходный CSV и docs/data_dictionary.md')

print('pandas:', pd.__version__)
print('Датасет:', data_path.name)


pandas: 3.0.6
Датасет: go_arounds_augmented.csv.gz


In [2]:
# Роль поля задаем явно: так легче проверить, что случайно не попало в модель
target_column = 'has_ga'
time_column = 'time'
identifier_columns = ['icao24', 'callsign', 'registration']
excluded_columns = ['runway', 'n_approaches', 'n_rwy_approached']

categorical_features = ['airport', 'typecode', 'wtc']
numeric_features = [
    'wind_speed_knts', 'visibility_m', 'temperature_deg', 'press_p',
]
candidate_features = categorical_features + numeric_features
deferred_columns = [
    'icaoaircrafttype', 'glide_slope_angle', 'has_intersection', 'rwy_length',
    'airport_country', 'airport_region', 'operator_country', 'operator_region',
    'wind_dir_deg', 'wind_gust_knts', 'press_sea_level_p',
    'weather_intensity', 'weather_precipitation', 'weather_desc',
    'weather_obscuration', 'weather_other',
]

# Проверяем только заголовок: для контроля схемы не нужна полная загрузка
columns = pd.read_csv(data_path, nrows=0).columns.tolist()
assigned_columns = (
    [target_column, time_column] + identifier_columns + excluded_columns
    + candidate_features + deferred_columns
)
if len(assigned_columns) != len(set(assigned_columns)):
    raise ValueError('Одно поле получило несколько ролей')
if set(columns) != set(assigned_columns):
    raise ValueError('Схема CSV отличается от словаря ролей')

documented_columns = [
    line.split('|')[1].strip().strip('`')
    for line in dictionary_path.read_text(encoding='utf-8').splitlines()
    if line.startswith('| `')
]
if documented_columns != columns:
    raise ValueError('Словарь не соответствует порядку полей исходного CSV')

print('Все поля получили роль:', len(columns))
print('Категориальные кандидаты:', categorical_features)
print('Числовые кандидаты:', numeric_features)


Все поля получили роль: 31
Категориальные кандидаты: ['airport', 'typecode', 'wtc']
Числовые кандидаты: ['wind_speed_knts', 'visibility_m', 'temperature_deg', 'press_p']


## Почему часть полей исключена

`n_approaches` и `n_rwy_approached` описывают последовательность заходов,
а `runway` относится к состоявшейся посадке. Поэтому они могут раскрывать
исход. Связанные с итоговой ВПП характеристики также откладываем.
Идентификаторы не даем модели как признаки, чтобы не поощрять запоминание.

Семь выбранных полей – кандидаты, а не окончательная схема обучения.
Принадлежность к кандидату не подтверждает доступность поля до события.


In [3]:
# Это только примеры значений, а не репрезентативная выборка
preview = pd.read_csv(data_path, nrows=10_000, low_memory=False)
feature_preview = pd.DataFrame({
    'dtype': preview[candidate_features].dtypes.astype(str),
    'examples': {
        column: preview[column].dropna().unique()[:5].tolist()
        for column in candidate_features
    },
})
display(feature_preview)


                   dtype                                                        examples
airport              str                                                          [KAUS]
typecode             str                                  [A319, A320, A321, B38M, E75L]
wtc                  str                                                  [M, L, H, L/M]
wind_speed_knts  float64                                     [0.0, 4.0, 7.0, 15.0, 17.0]
visibility_m     float64             [16093.44, 14484.096, 12874.752, 4828.032, 8046.72]
temperature_deg  float64                                      [10.0, 7.2, 6.7, 4.4, 2.2]
press_p          float64  [1014.901083, 1015.578361, 1015.917, 1016.594278, 1017.610195]

## Вывод

Все 31 поле описаны и получили ровно одну роль. Схема словаря проверена
по заголовку CSV. Выбраны семь кандидатов: три категориальных и четыре числовых.
Примеры значений взяты из первых 10 000 строк, не из случайной выборки.

Итоговая ВПП, число попыток и идентификаторы не входят в первый набор признаков.
Проверка аномалий вынесена в [03_data_quality.ipynb](03_data_quality.ipynb).
До обучения нужно принять решение об обработке. Связь METAR с моментом
события и отдельные кодировки остаются открытыми вопросами.

Разбиение train/validation/test и обучение пока не выполнены.
